# Módulos estándar · Serialización y red

Guardar objetos de Python en bytes y recuperarlos (`pickle`, `shelve`, `struct`) y comunicar programas por la red con la biblioteca estándar: *sockets* TCP y UDP, `socketserver`, y un servidor HTTP con `http.server` consultado con `urllib`. Los servidores se lanzan en hilos dentro del notebook y escuchan solo en `127.0.0.1`, en un puerto libre que elige el sistema operativo.

## 1. `pickle` frente a JSON

Crear un diccionario con tipos que JSON no admite (un conjunto, una tupla, una fecha y un número complejo). Serializarlo con `pickle.dumps`, recuperarlo con `pickle.loads` y comprobar que es igual al original y conserva los tipos. Intentar lo mismo con `json.dumps` y capturar el error.

In [1]:
import json
import pickle
from datetime import date

datos = {
    "islas": {"Tenerife", "Gran Canaria", "Lanzarote"},
    "coordenadas": (28.29, -16.63),
    "fecha": date(2026, 5, 30),
    "impedancia": 3 + 4j,
    "notas": [7.5, 9, None],
}
binario = pickle.dumps(datos)
recuperado = pickle.loads(binario)
print(len(binario), "bytes:", binario[:40], "...")
print({k: type(v).__name__ for k, v in recuperado.items()})
assert recuperado == datos and recuperado is not datos

try:
    json.dumps(datos)
except TypeError as e:
    print("JSON no puede:", e)

221 bytes: b'\x80\x04\x95\xd2\x00\x00\x00\x00\x00\x00\x00}\x94(\x8c\x05islas\x94\x8f\x94(\x8c\tLanzarote\x94\x8c\x08T' ...
{'islas': 'set', 'coordenadas': 'tuple', 'fecha': 'date', 'impedancia': 'complex', 'notas': 'list'}
JSON no puede: Object of type set is not JSON serializable


## 2. Varios objetos en un fichero

Guardar en `registros.pkl` cinco registros seguidos con `pickle.dump` y leerlos después uno a uno con `pickle.load` hasta que se acabe el fichero (`EOFError`). Comparar el tamaño de un mismo objeto con los protocolos 0 a `pickle.HIGHEST_PROTOCOL`.

In [2]:
from pathlib import Path

registros = [{"id": i, "nombre": f"cliente {i}", "saldo": 100.0 * i} for i in range(1, 6)]
with open("registros.pkl", "wb") as f:
    for r in registros:
        pickle.dump(r, f)

leidos = []
with open("registros.pkl", "rb") as f:
    while True:
        try:
            leidos.append(pickle.load(f))
        except EOFError:
            break
print(len(leidos), "registros leídos de", Path("registros.pkl").stat().st_size, "bytes")
assert leidos == registros

for protocolo in range(pickle.HIGHEST_PROTOCOL + 1):
    print(f"protocolo {protocolo}: {len(pickle.dumps(registros, protocol=protocolo))} bytes")

5 registros leídos de 305 bytes
protocolo 0: 239 bytes
protocolo 1: 224 bytes
protocolo 2: 226 bytes
protocolo 3: 226 bytes
protocolo 4: 197 bytes
protocolo 5: 197 bytes


## 3. Controlar qué se serializa

Crear una clase `Contador` que guarde un total y un `threading.Lock` (que no se puede serializar). Comprobar que `pickle` falla con la clase tal cual y añadir `__getstate__` y `__setstate__` para guardar solo el total y crear un cerrojo nuevo al cargar.

In [3]:
import threading


class ContadorSimple:
    def __init__(self):
        self.total = 0
        self.cerrojo = threading.Lock()


class Contador(ContadorSimple):
    def incrementar(self, n=1):
        with self.cerrojo:
            self.total += n

    def __getstate__(self):
        return {"total": self.total}

    def __setstate__(self, estado):
        self.total = estado["total"]
        self.cerrojo = threading.Lock()


try:
    pickle.dumps(ContadorSimple())
except TypeError as e:
    print("Sin __getstate__:", e)

c = Contador()
c.incrementar(5)
copia = pickle.loads(pickle.dumps(c))
copia.incrementar()
print(copia.total, type(copia.cerrojo).__name__)
assert (c.total, copia.total) == (5, 6) and copia.cerrojo is not c.cerrojo

Sin __getstate__: cannot pickle '_thread.lock' object
6 lock


## 4. Cargar `pickle` ejecuta código

`pickle` no es un formato de datos seguro: al cargar se pueden ejecutar funciones arbitrarias. Crear una clase cuyo `__reduce__` haga que al deserializarla se llame a `print` (inofensivo) y comprobar que el mensaje aparece al hacer `loads`. Por eso nunca se debe cargar un *pickle* de origen desconocido.

In [4]:
class Trampa:
    def __reduce__(self):
        return (print, ("¡Esto se ha ejecutado al cargar el pickle!",))


carga = pickle.dumps(Trampa())
print("Bytes recibidos:", carga)
resultado = pickle.loads(carga)
print("loads devolvió:", resultado)

Bytes recibidos: b'\x80\x04\x95H\x00\x00\x00\x00\x00\x00\x00\x8c\x08builtins\x94\x8c\x05print\x94\x93\x94\x8c+\xc2\xa1Esto se ha ejecutado al cargar el pickle!\x94\x85\x94R\x94.'
¡Esto se ha ejecutado al cargar el pickle!
loads devolvió: None


## 5. Diccionarios persistentes con `shelve`

Guardar una agenda en un `shelve` (un diccionario que vive en disco y guarda los valores con `pickle`). Cerrarlo, volver a abrirlo para añadir un contacto y modificar otro, y comprobar que los cambios se han guardado. Mostrar por qué hace falta reasignar la clave (o `writeback=True`) al modificar un valor mutable.

In [5]:
import shelve

with shelve.open("agenda") as agenda:
    agenda["ana"] = {"telefono": "928 000 001", "etiquetas": ["trabajo"]}
    agenda["luis"] = {"telefono": "922 000 002", "etiquetas": []}

with shelve.open("agenda") as agenda:
    agenda["luis"]["etiquetas"].append("familia")  # se modifica una copia que no se guarda
    print("Sin reasignar:", agenda["luis"]["etiquetas"])
    contacto = agenda["luis"]
    contacto["etiquetas"].append("familia")
    agenda["luis"] = contacto
    agenda["carmen"] = {"telefono": "928 000 003", "etiquetas": ["club"]}

with shelve.open("agenda", writeback=True) as agenda:
    agenda["ana"]["etiquetas"].append("pádel")

with shelve.open("agenda", flag="r") as agenda:
    for nombre in sorted(agenda):
        print(nombre, agenda[nombre])
    assert agenda["luis"]["etiquetas"] == ["familia"] and agenda["ana"]["etiquetas"] == ["trabajo", "pádel"]

Sin reasignar: []
ana {'telefono': '928 000 001', 'etiquetas': ['trabajo', 'pádel']}
carmen {'telefono': '928 000 003', 'etiquetas': ['club']}
luis {'telefono': '922 000 002', 'etiquetas': ['familia']}


## 6. Registros binarios con `struct`

Definir con `struct` un registro de tamaño fijo (identificador entero, precio en coma flotante y nombre de 12 bytes), escribir varios productos en `productos.bin`, leer directamente el tercero saltando con `seek` y leerlos todos con `iter_unpack`. Comparar el orden de bytes *little-endian* y *big-endian* de un mismo entero.

In [6]:
import struct

FORMATO = struct.Struct("<id12s")
productos = [(1, 1.25, "gofio"), (2, 3.9, "mojo picón"), (3, 12.5, "queso"), (4, 2.0, "papas")]
with open("productos.bin", "wb") as f:
    for ident, precio, nombre in productos:
        f.write(FORMATO.pack(ident, precio, nombre.encode("utf-8")))

print("Tamaño del registro:", FORMATO.size, "bytes; fichero:", Path("productos.bin").stat().st_size, "bytes")
with open("productos.bin", "rb") as f:
    f.seek(2 * FORMATO.size)
    ident, precio, nombre = FORMATO.unpack(f.read(FORMATO.size))
    print("Tercero:", ident, precio, nombre.rstrip(b"\0").decode("utf-8"))
    f.seek(0)
    todos = [(i, p, n.rstrip(b"\0").decode("utf-8")) for i, p, n in FORMATO.iter_unpack(f.read())]
assert todos == productos
print(struct.pack("<I", 1000).hex(" "), "|", struct.pack(">I", 1000).hex(" "))

Tamaño del registro: 24 bytes; fichero: 96 bytes
Tercero: 3 12.5 queso
e8 03 00 00 | 00 00 03 e8


## 7. Servidor y cliente TCP

Programar con `socket` un servidor de eco TCP que atienda a un cliente: recibe líneas y devuelve cada una en mayúsculas hasta recibir `FIN`. Lanzarlo en un hilo escuchando en `127.0.0.1` y en un puerto libre (puerto 0), conectar un cliente, enviar tres mensajes y mostrar las respuestas.

In [7]:
import socket


def servidor_eco(escucha):
    conexion, direccion = escucha.accept()
    with conexion, conexion.makefile("rw", encoding="utf-8", newline="\n") as canal:
        for linea in canal:
            if linea.strip() == "FIN":
                break
            canal.write(linea.upper())
            canal.flush()


escucha = socket.create_server(("127.0.0.1", 0))
puerto = escucha.getsockname()[1]
hilo = threading.Thread(target=servidor_eco, args=(escucha,))
hilo.start()

respuestas = []
with socket.create_connection(("127.0.0.1", puerto)) as cliente, cliente.makefile("rw", encoding="utf-8", newline="\n") as canal:
    for mensaje in ["hola", "papas arrugadas", "¿qué tal?"]:
        canal.write(mensaje + "\n")
        canal.flush()
        respuestas.append(canal.readline().strip())
    canal.write("FIN\n")
    canal.flush()
hilo.join()
escucha.close()
print(respuestas)
assert respuestas == ["HOLA", "PAPAS ARRUGADAS", "¿QUÉ TAL?"]

['HOLA', 'PAPAS ARRUGADAS', '¿QUÉ TAL?']


## 8. Varios clientes con `socketserver`

Reescribir el servidor con `socketserver.ThreadingTCPServer`, que atiende cada conexión en su propio hilo, para que calcule la suma de los números que recibe en cada línea. Conectar cinco clientes a la vez desde un `ThreadPoolExecutor` y comprobar todas las respuestas. Apagar el servidor con `shutdown`.

In [8]:
import socketserver
from concurrent.futures import ThreadPoolExecutor


class Sumador(socketserver.StreamRequestHandler):
    def handle(self):
        for linea in self.rfile:
            numeros = [float(x) for x in linea.decode().split()]
            self.wfile.write(f"{sum(numeros):g}\n".encode())


def cliente(i):
    with socket.create_connection(("127.0.0.1", puerto)) as s, s.makefile("rwb") as canal:
        canal.write(" ".join(str(n) for n in range(i + 1)).encode() + b"\n")
        canal.flush()
        return canal.readline().decode().strip()


with socketserver.ThreadingTCPServer(("127.0.0.1", 0), Sumador) as servidor:
    puerto = servidor.server_address[1]
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    with ThreadPoolExecutor(5) as pool:
        resultados = list(pool.map(cliente, range(1, 6)))
    servidor.shutdown()
print(resultados)
assert resultados == [str(sum(range(i + 1))) for i in range(1, 6)]

['1', '3', '6', '10', '15']


## 9. Datagramas UDP

UDP envía mensajes sueltos sin conexión ni garantía de entrega. Crear un servidor UDP que responda a cada datagrama con su longitud y un cliente que envíe tres mensajes con `sendto` y lea las respuestas con `recvfrom` (con un tiempo de espera máximo).

In [9]:
def servidor_udp(s, n):
    for _ in range(n):
        datos, origen = s.recvfrom(1024)
        s.sendto(str(len(datos.decode())).encode(), origen)


udp = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
udp.bind(("127.0.0.1", 0))
mensajes = ["Teide", "Roque Nublo", "Timanfaya"]
hilo = threading.Thread(target=servidor_udp, args=(udp, len(mensajes)))
hilo.start()

with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as cliente_udp:
    cliente_udp.settimeout(2)
    longitudes = []
    for m in mensajes:
        cliente_udp.sendto(m.encode(), udp.getsockname())
        longitudes.append(int(cliente_udp.recvfrom(1024)[0]))
hilo.join()
udp.close()
print(dict(zip(mensajes, longitudes)))
assert longitudes == [len(m) for m in mensajes]

{'Teide': 5, 'Roque Nublo': 11, 'Timanfaya': 9}


## 10. HTTP con `http.server` y `urllib`

Crear con `http.server` una pequeña API que responda en JSON a `GET /islas?min=N` con las islas de más de N km² y con un error 404 a cualquier otra ruta. Lanzarla en un hilo y consultarla con `urllib.request`, construyendo la consulta con `urllib.parse.urlencode`. Capturar el `HTTPError` de la ruta inexistente.

In [10]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.error import HTTPError
from urllib.parse import parse_qs, urlencode, urlparse
from urllib.request import urlopen

SUPERFICIES = {"Tenerife": 2034, "Fuerteventura": 1660, "Gran Canaria": 1560, "Lanzarote": 846, "La Palma": 708, "La Gomera": 370, "El Hierro": 269}


class Api(BaseHTTPRequestHandler):
    def do_GET(self):
        url = urlparse(self.path)
        if url.path != "/islas":
            self.send_error(404, "Ruta no encontrada")
            return
        minimo = int(parse_qs(url.query).get("min", ["0"])[0])
        cuerpo = json.dumps({isla: km2 for isla, km2 in SUPERFICIES.items() if km2 > minimo}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(cuerpo)))
        self.end_headers()
        self.wfile.write(cuerpo)

    def log_message(self, *args):
        pass


with ThreadingHTTPServer(("127.0.0.1", 0), Api) as http:
    base = f"http://127.0.0.1:{http.server_address[1]}"
    threading.Thread(target=http.serve_forever, daemon=True).start()
    with urlopen(f"{base}/islas?{urlencode({'min': 1000})}") as respuesta:
        print(respuesta.status, respuesta.headers["Content-Type"], json.loads(respuesta.read()))
    try:
        urlopen(f"{base}/volcanes")
    except HTTPError as e:
        print("Error:", e.code, e.reason)
    http.shutdown()

200 application/json {'Tenerife': 2034, 'Fuerteventura': 1660, 'Gran Canaria': 1560}
Error: 404 Ruta no encontrada
